# 02 · Data Cleaning
**Goal:** turn the raw CSV into an analysis-ready file in `data/processed/`. The raw file is read-only: it is **never modified**.

Cleaning rules (decided in notebook 01):
1. Drop the two empty unnamed columns.
2. Remove exact duplicate rows.
3. Fill missing `placement` with `Unknown`.
4. Fix data types and add a `month_number` column for correct month ordering.
5. Add **flag columns** for suspicious rows instead of deleting them.
6. Check outliers but keep them (they are valid large campaigns).

In [1]:
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd

RAW_PATH = Path("..") / "data" / "raw" / "online_advertising_performance_data.csv"
OUT_PATH = Path("..") / "data" / "processed" / "advertising_cleaned.csv"

def file_hash(path):
    return hashlib.md5(path.read_bytes()).hexdigest()

raw_hash_before = file_hash(RAW_PATH)
raw = pd.read_csv(RAW_PATH)
print("Raw shape:", raw.shape)

Raw shape: (15408, 14)


## 1. Drop empty columns

In [2]:
empty_cols = [c for c in raw.columns if raw[c].isna().all()]
print("Dropping:", empty_cols)
df = raw.drop(columns=empty_cols).copy()
df.shape

Dropping: ['Unnamed: 12', 'Unnamed: 13']


(15408, 12)

## 2. Remove exact duplicates

In [3]:
rows_before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Removed {rows_before - len(df)} duplicate rows -> {len(df):,} rows remain")

Removed 5 duplicate rows -> 15,403 rows remain


## 3. Missing `placement`

In [4]:
print("Missing before:", df["placement"].isna().sum())
df["placement"] = df["placement"].fillna("Unknown")
print("Missing after: ", df["placement"].isna().sum())

Missing before: 413
Missing after:  0


Why `Unknown` and not deletion: those 413 rows contain real cost, clicks and revenue. Deleting them would understate totals. Filling with `Unknown` keeps the money and clicks while making it clear the placement is not recorded.

## 4. Clean text and set data types

In [5]:
text_cols = ["month", "campaign_number", "user_engagement", "banner", "placement"]
for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

month_order = ["April", "May", "June"]
month_numbers = {"April": 4, "May": 5, "June": 6}
df["month_number"] = df["month"].map(month_numbers).astype("int64")

# Ordered categories give correct sorting in charts
df["month"] = pd.Categorical(df["month"], categories=month_order, ordered=True)
df["user_engagement"] = pd.Categorical(df["user_engagement"], categories=["Low", "Medium", "High"], ordered=True)

df["day"] = df["day"].astype("int64")
count_cols = ["displays", "clicks", "post_click_conversions"]
money_cols = ["cost", "revenue", "post_click_sales_amount"]
df[count_cols] = df[count_cols].astype("int64")
df[money_cols] = df[money_cols].astype("float64")
df.dtypes

month                      category
day                           int64
campaign_number                 str
user_engagement            category
banner                          str
placement                       str
displays                      int64
cost                        float64
clicks                        int64
revenue                     float64
post_click_conversions        int64
post_click_sales_amount     float64
month_number                  int64
dtype: object

## 5. Flag suspicious rows (rows are kept)

In [6]:
df["flag_clicks_gt_displays"] = df["clicks"] > df["displays"]
df["flag_conversions_gt_clicks"] = df["post_click_conversions"] > df["clicks"]
df["flag_cost_without_displays"] = (df["cost"] > 0) & (df["displays"] == 0)
df["flag_displays_without_cost"] = (df["displays"] > 0) & (df["cost"] == 0)
flag_cols = [c for c in df.columns if c.startswith("flag_")]
df[flag_cols] = df[flag_cols].astype(int)   # 0/1 is easiest to use in Power BI
df[flag_cols].sum()

flag_clicks_gt_displays         2
flag_conversions_gt_clicks    328
flag_cost_without_displays     15
flag_displays_without_cost     57
dtype: int64

## 6. Outlier check (nothing is removed)

In [7]:
summary = []
for col in ["displays", "cost", "clicks", "revenue", "post_click_conversions", "post_click_sales_amount"]:
    values = df.loc[df[col] > 0, col]
    q1, q3 = values.quantile([0.25, 0.75])
    upper = q3 + 1.5 * (q3 - q1)
    n_out = int((values > upper).sum())
    summary.append({"column": col, "non_zero_rows": len(values), "upper_fence_1.5IQR": upper,
                    "rows_above_fence": n_out, "share_%": n_out / len(values) * 100, "max": values.max()})
pd.DataFrame(summary).round(2)

,column,non_zero_rows,upper_fence_1.5IQR,rows_above_fence,share_%,max
0,displays,15381,22416.50,2245,14.60,455986.00
1,cost,15339,6.35,2494,16.26,556.70
2,clicks,10511,262.50,1423,13.54,14566.00
3,revenue,10305,21.78,1661,16.12,2096.21
4,post_click_conversions,5568,104.50,903,16.22,3369.00
5,post_click_sales_amount,5587,5822.04,886,15.86,199930.32


In [8]:
# Are the biggest rows realistic? Check how much of the total the top 1% of rows account for.
top_n = int(len(df) * 0.01)
top_rows = df.nlargest(top_n, "cost")
print(f"Top 1% of rows by cost ({top_n} rows) = {top_rows['cost'].sum() / df['cost'].sum():.1%} of total cost")
top_rows[["month", "day", "campaign_number", "user_engagement", "banner", "placement",
          "displays", "cost", "clicks", "revenue"]].head(8)

Top 1% of rows by cost (154 rows) = 30.6% of total cost


,month,day,campaign_number,user_engagement,banner,placement,displays,cost,clicks,revenue
7973,May,12,camp 1,High,300 x 250,mno,383436,556.7048,2178,406.0
6902,May,5,camp 1,High,300 x 250,mno,369042,553.0831,2052,378.0
6747,May,4,camp 1,High,300 x 250,mno,375291,539.1284,1955,355.0
8889,May,18,camp 1,High,300 x 250,mno,347923,515.8464,1935,369.0
7820,May,11,camp 1,High,300 x 250,mno,373289,513.3684,2012,372.0
8732,May,17,camp 1,High,300 x 250,mno,356985,501.9376,1994,379.0
6595,May,3,camp 1,High,300 x 250,mno,331455,485.5104,2428,456.0
8128,May,13,camp 1,High,300 x 250,mno,342398,484.8313,1783,338.0


**Decision:** the large values are consistent with each other (high displays go with high cost and high clicks), so they are valid high-volume placements, not errors. Because the data is so skewed, notebooks 03 and 04 compare **totals and ratios of totals** (for example total cost / total clicks) rather than averaging row-level ratios.

## 7. Reconcile totals and save

In [9]:
value_cols = ["displays", "cost", "clicks", "revenue", "post_click_conversions", "post_click_sales_amount"]
reconcile = pd.DataFrame({
    "raw_total": raw[value_cols].sum(),
    "cleaned_total": df[value_cols].sum(),
})
reconcile["difference"] = reconcile["raw_total"] - reconcile["cleaned_total"]
reconcile

,raw_total,cleaned_total,difference
displays,2.390177e+08,2.390143e+08,3464.0000
cost,1.751930e+05,1.751922e+05,0.7757
clicks,2.492837e+06,2.492825e+06,12.0000
revenue,2.762646e+05,2.762626e+05,1.9462
post_click_conversions,6.517680e+05,6.517620e+05,6.0000
post_click_sales_amount,3.271562e+07,3.271541e+07,212.1718


The small difference is exactly the 5 duplicate rows that were removed. Nothing else changed.

In [10]:
df.to_csv(OUT_PATH, index=False)

check = pd.read_csv(OUT_PATH)
print("Saved:", OUT_PATH, "| shape:", check.shape)
print("Missing values in cleaned file:", int(check.isna().sum().sum()))
print("Raw file unchanged:", raw_hash_before == file_hash(RAW_PATH))
check.head()

Saved: ../data/processed/advertising_cleaned.csv | shape: (15403, 17)
Missing values in cleaned file: 0
Raw file unchanged: True


,month,day,campaign_number,user_engagement,banner,placement,displays,cost,clicks,revenue,post_click_conversions,post_click_sales_amount,month_number,flag_clicks_gt_displays,flag_conversions_gt_clicks,flag_cost_without_displays,flag_displays_without_cost
0,April,1,camp 1,High,160 x 600,abc,4,0.0060,0,0.0000,0,0.0000,4,0,0,0,0
1,April,1,camp 1,High,160 x 600,def,20170,26.7824,158,28.9717,23,1972.4602,4,0,0,0,0
2,April,1,camp 1,High,160 x 600,ghi,14701,27.6304,158,28.9771,78,2497.2636,4,0,0,0,0
3,April,1,camp 1,High,160 x 600,mno,171259,216.8750,1796,329.4518,617,24625.3234,4,0,0,0,0
4,April,1,camp 1,Low,160 x 600,def,552,0.0670,1,0.1834,0,0.0000,4,0,0,0,0


## Cleaning summary
- **Raw:** 15,408 rows x 14 columns → **Cleaned:** 15,403 rows x 17 columns (12 original columns + `month_number` + 4 flag columns).
- Removed: 2 empty columns, 5 duplicate rows.
- Filled: 413 missing `placement` values with `Unknown`.
- Kept and flagged: suspicious rows. Kept: all outliers.
- Saved to `data/processed/advertising_cleaned.csv`; raw file verified unchanged.